# SeamlessM4T v2 Large: Structured Compression 2.3B to ~1B

## Compression Pipeline
| Phase | Technique | Paper | Expected |
|-------|-----------|-------|----------|
| 0 | Baseline benchmark | - | reference |
| 1 | Vocabulary/Embedding pruning | Asahi (EMNLP 2023) | -200M |
| 2 | Text encoder removal (S2S-only) | Architecture analysis | -350M |
| 3 | Text decoder iterative layer pruning | Moslem (IWSLT 2025) | -150M |
| 4 | Speech encoder iterative layer pruning | ShortGPT (ACL 2025) | -150M |
| 5 | Width pruning (FLAP on FFN + heads) | FLAP (AAAI 2024) | -200M |
| 6 | T2U model pruning | Iterative layer pruning | -50M |
| 7 | Recovery fine-tuning (LoRA + S2TT CE) | Moslem (IWSLT 2025) | quality up |
| 8 | Final benchmark + paper table | - | - |

## Setup Cells 1-8
Run these at the **start of every** Kaggle session.

In [ ]:
import os, sys, subprocess, pathlib, re, glob, json, gc, copy, time, math
import warnings; warnings.filterwarnings('ignore')

ON_KAGGLE = os.path.exists('/kaggle/working')
PLATFORM  = 'kaggle' if ON_KAGGLE else 'colab'
WORK_DIR  = '/kaggle/working' if ON_KAGGLE else '/content'
CKPT_DIR  = f'{WORK_DIR}/checkpoints'
AUDIO_DIR = f'{WORK_DIR}/audio'
FIG_DIR   = f'{WORK_DIR}/figures'
MODEL_DIR = f'{WORK_DIR}/models'
for d in [CKPT_DIR, AUDIO_DIR, FIG_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Platform : {PLATFORM}')
print(f'Work dir : {WORK_DIR}')

In [ ]:
subprocess.run('curl -s https://rclone.org/install.sh | sudo bash',
               shell=True, capture_output=True)
ver = subprocess.run('rclone version', shell=True, capture_output=True, text=True)
print(ver.stdout.split(chr(10))[0])

In [ ]:
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    RCLONE_CONF = UserSecretsClient().get_secret('RCLONE_CONF')
else:
    from google.colab import userdata
    RCLONE_CONF = userdata.get('RCLONE_CONF')

raw = RCLONE_CONF.strip()
raw = re.sub(r'\s*(\[[^\]]+\])\s*', r'\n\1\n', raw)
raw = re.sub(r'\s+(type|scope|token|team_drive|client_id|client_secret|'
             r'root_folder_id|service_account_file|drive_id)\s*=\s*',
             r'\n\1 = ', raw)
raw = raw.strip() + chr(10)

rclone_cfg = pathlib.Path.home() / '.config/rclone/rclone.conf'
rclone_cfg.parent.mkdir(parents=True, exist_ok=True)
rclone_cfg.write_text(raw)
r = subprocess.run('rclone lsd gdrive:', shell=True, capture_output=True, text=True)
print('Drive root:' if r.returncode == 0 else 'FAILED:')
print(r.stdout[:300] or r.stderr[:300])

In [ ]:
subprocess.run([
    'pip', 'install', '-q',
    'transformers', 'datasets', 'torchaudio', 'speechbrain',
    'peft', 'librosa', 'jiwer', 'evaluate', 'sacrebleu',
    'sentencepiece', 'accelerate', 'matplotlib', 'seaborn',
], check=True)
print('All packages installed.')

In [ ]:
print('Pulling checkpoints from Google Drive...')
subprocess.run(f'rclone sync gdrive:cse465v3/checkpoints {CKPT_DIR}',
               shell=True, capture_output=True, text=True)
files = sorted(os.listdir(CKPT_DIR)) if os.path.isdir(CKPT_DIR) else []
if files:
    print(f'{len(files)} file(s) found:')
    for f in files:
        mb = os.path.getsize(f'{CKPT_DIR}/{f}') / 1e6
        print(f'  {f:<55} {mb:>8.1f} MB')
else:
    print('No checkpoints yet.')

In [ ]:
import torch
from datetime import datetime

GDRIVE_ROOT = 'gdrive:cse465v3'

def save_checkpoint(state, name, step=0, keep=3):
    fname = f'{name}_step{step:06d}.pt'
    local = f'{CKPT_DIR}/{fname}'
    torch.save(state, local)
    mb = os.path.getsize(local) / 1e6
    print(f'[ckpt] Saved {fname} ({mb:.1f} MB)')
    subprocess.run(f'rclone copy {local} {GDRIVE_ROOT}/checkpoints/',
                   shell=True, capture_output=True, text=True)
    old = sorted(glob.glob(f'{CKPT_DIR}/{name}_step*.pt'))
    for f in old[:-keep]: os.remove(f)

def load_latest_checkpoint(name):
    files = sorted(glob.glob(f'{CKPT_DIR}/{name}_step*.pt'))
    if not files:
        print(f'[ckpt] No checkpoint for {name!r}')
        return None
    state = torch.load(files[-1], map_location='cpu', weights_only=False)
    print(f'[ckpt] Loaded {os.path.basename(files[-1])}')
    return state

def save_model_to_drive(mdl, proc, stage_name):
    local = f'{MODEL_DIR}/{stage_name}'
    os.makedirs(local, exist_ok=True)
    print(f'[model] Saving {stage_name}...')
    mdl.save_pretrained(local)
    proc.save_pretrained(local)
    total = sum(os.path.getsize(f'{local}/{f}') for f in os.listdir(local)) / 1e6
    print(f'[model] Size: {total:.0f} MB  uploading...')
    subprocess.run(f'rclone sync {local} {GDRIVE_ROOT}/{stage_name}/',
                   shell=True, capture_output=True, text=True)
    print('[model] Done.')

def load_model_from_drive(stage_name):
    from transformers import SeamlessM4Tv2ForSpeechToSpeech, SeamlessM4TProcessor
    local = f'{MODEL_DIR}/{stage_name}'
    if not (os.path.exists(local) and os.listdir(local)):
        print(f'[model] Downloading {stage_name} from Drive...')
        os.makedirs(local, exist_ok=True)
        subprocess.run(f'rclone sync {GDRIVE_ROOT}/{stage_name}/ {local}', shell=True)
    print(f'[model] Loading {stage_name}...')
    mdl = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
        local, torch_dtype=torch.float16, device_map='auto')
    proc = SeamlessM4TProcessor.from_pretrained(local)
    mdl.eval()
    return mdl, proc

print('I/O functions ready.')

In [ ]:
import torchaudio, numpy as np
from IPython.display import Audio as IPAudio, display

def play(audio, sr, label=''):
    if hasattr(audio, 'numpy'): audio = audio.squeeze().numpy()
    print(f'  {label}  ({len(audio)/sr:.1f}s | sr={sr})')
    display(IPAudio(audio, rate=int(sr)))

def save_audio(audio, sr, filename, label=''):
    path = f'{AUDIO_DIR}/{filename}'
    if hasattr(audio, 'numpy'): t = audio.squeeze().unsqueeze(0).float()
    else: t = torch.tensor(audio).unsqueeze(0).float()
    torchaudio.save(path, t, sr)
    print(f'[audio] {filename} ({os.path.getsize(path)/1e6:.1f} MB)')
    subprocess.run(f'rclone copy {path} {GDRIVE_ROOT}/audio/',
                   shell=True, capture_output=True, text=True)

print('Audio helpers ready.')

In [ ]:
def session_status():
    print('=' * 60)
    print(f'  Platform : {PLATFORM}   Time : {datetime.now():%Y-%m-%d %H:%M}')
    local = [f for f in glob.glob(f'{CKPT_DIR}/**', recursive=True) if os.path.isfile(f)]
    print(f'  Local checkpoint files: {len(local)}')
    for f in local[:20]:
        print(f'    {os.path.relpath(f, CKPT_DIR):<50} {os.path.getsize(f)/1e6:>8.1f} MB')
    if torch.cuda.is_available():
        print(f'  GPU: {torch.cuda.get_device_name(0)}')
        print(f'  VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')
    print('=' * 60)
session_status()

## Core Library: Model, Benchmark, Plotting

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt, matplotlib, seaborn as sns
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120, 'savefig.bbox': 'tight'})
sns.set_style('whitegrid')

def count_params(module):
    return sum(p.numel() for p in module.parameters()) / 1e6

def count_params_detailed(model):
    bd = {}
    for name, child in model.named_children():
        bd[name] = count_params(child)
    bd['TOTAL'] = count_params(model)
    return bd

def print_model_breakdown(model, title='Model Breakdown'):
    bd = count_params_detailed(model)
    print(f'\n--- {title} ---')
    total = bd.pop('TOTAL')
    for name, p in sorted(bd.items(), key=lambda x: -x[1]):
        pct = p / total * 100 if total > 0 else 0
        print(f'  {name:<35} {p:>8.1f}M  ({pct:>5.1f}%)')
    print(f'  {"TOTAL":<35} {total:>8.1f}M')
    print('---')
    return {**bd, 'TOTAL': total}

def gpu_mem():
    if torch.cuda.is_available():
        a = torch.cuda.memory_allocated() / 1e9
        r = torch.cuda.memory_reserved() / 1e9
        print(f'  GPU mem: {a:.2f} GB alloc / {r:.2f} GB reserved')

print('Core utilities ready.')

In [ ]:
from transformers import SeamlessM4Tv2ForSpeechToSpeech, SeamlessM4TProcessor

if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret('HF_TOKEN'))
    print('Logged into HuggingFace Hub.')

MODEL_NAME = 'facebook/seamless-m4t-v2-large'

def load_base_model():
    print(f'Loading processor from {MODEL_NAME}...')
    proc = SeamlessM4TProcessor.from_pretrained(MODEL_NAME)
    print(f'Loading model  -- may take 5-10 min...')
    mdl = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
    mdl.eval()
    print('Model loaded.'); gpu_mem()
    return mdl, proc

print('load_base_model() ready.')

In [ ]:
from sacrebleu.metrics import BLEU, CHRF

_bleu = BLEU(effective_order=True)
_chrf = CHRF()

def compute_bleu(hyp, ref):
    if not hyp.strip() or not ref.strip(): return 0.0
    return _bleu.sentence_score(hyp.strip(), [ref.strip()]).score

def compute_chrf(hyp, ref):
    if not hyp.strip() or not ref.strip(): return 0.0
    return _chrf.sentence_score(hyp.strip(), [ref.strip()]).score

def run_s2st(mdl, wav, tgt_lang='ben'):
    """Single forward pass returning (text, audio_numpy)."""
    inputs = processor(audios=wav, sampling_rate=16000, return_tensors='pt')
    inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mdl.generate(**inputs, tgt_lang=tgt_lang,
                           return_intermediate_token_ids=True)
    text_ids = out.sequences.cpu()
    text = processor.batch_decode(text_ids, skip_special_tokens=True)[0]
    wav_out = out.waveform.cpu().numpy().squeeze() if out.waveform is not None else np.zeros(16000)
    return text, wav_out

def run_s2t_only(mdl, wav, tgt_lang='ben'):
    """Text-only generation for fast eval during iterative pruning."""
    inputs = processor(audios=wav, sampling_rate=16000, return_tensors='pt')
    inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
    with torch.no_grad():
        text_ids = mdl.generate(**inputs, tgt_lang=tgt_lang, generate_speech=False)
    return processor.batch_decode(text_ids, skip_special_tokens=True)[0]

def run_benchmark(mdl, samples, label='model', tgt_lang='ben', save_n=0):
    """Full benchmark: BLEU, ChrF, RTF."""
    print(f'\n{"="*60}\n  BENCHMARK: {label}\n  Samples: {len(samples)}  Target: {tgt_lang}\n{"="*60}\n')
    results = []
    for i, s in enumerate(samples):
        try:
            dur = len(s['wav']) / 16000
            t0 = time.time()
            pred_text, out_wav = run_s2st(mdl, s['wav'], tgt_lang=tgt_lang)
            elapsed = time.time() - t0
            rtf  = elapsed / dur
            bleu = compute_bleu(pred_text, s['ref'])
            chrf = compute_chrf(pred_text, s['ref'])
            print(f'  [{i+1:>2}/{len(samples)}] BLEU={bleu:5.1f} ChrF={chrf:5.1f} RTF={rtf:.3f}  id={s["id"]}')
            if save_n > 0 and i < save_n:
                save_audio(out_wav, mdl.config.sampling_rate, f'{label}_s{i+1}.wav')
            results.append(dict(id=s['id'], bleu=bleu, chrf=chrf, rtf=rtf, pred=pred_text, ref=s['ref']))
        except Exception as e:
            print(f'  [{i+1:>2}/{len(samples)}] ERROR: {e}')
            results.append(dict(id=s['id'], bleu=0, chrf=0, rtf=float('nan'), pred='', ref=s.get('ref','')))
    valid = [r for r in results if not math.isnan(r['rtf'])]
    summary = dict(label=label, n=len(valid),
        avg_bleu=float(np.mean([r['bleu'] for r in valid])) if valid else 0,
        avg_chrf=float(np.mean([r['chrf'] for r in valid])) if valid else 0,
        avg_rtf=float(np.mean([r['rtf'] for r in valid])) if valid else 0,
        params_M=count_params(mdl))
    print(f'\n  Summary: BLEU={summary["avg_bleu"]:.2f}  ChrF={summary["avg_chrf"]:.2f}'
          f'  RTF={summary["avg_rtf"]:.4f}  Params={summary["params_M"]:.1f}M\n')
    return results, summary

def quick_eval_chrf(mdl, samples, tgt_lang='ben', max_samples=10):
    """Fast ChrF eval (text-only). For iterative pruning decisions."""
    scores = []
    for s in samples[:max_samples]:
        try: scores.append(compute_chrf(run_s2t_only(mdl, s['wav'], tgt_lang), s['ref']))
        except: scores.append(0.0)
    return float(np.mean(scores))

print('Benchmark functions ready.')

In [ ]:
ALL_SUMMARIES = []

def store_summary(s):
    ALL_SUMMARIES.append(s.copy())
    save_checkpoint({'summaries': ALL_SUMMARIES}, name='all_summaries', step=len(ALL_SUMMARIES))

def plot_phase_comparison(summaries=None, save_name='phase_comparison.png'):
    data = summaries or ALL_SUMMARIES
    if not data: print('No summaries yet.'); return
    labels = [s['label'] for s in data]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Compression Pipeline: Phase Comparison', fontsize=15, fontweight='bold')
    metrics = [('avg_bleu', 'BLEU (higher=better)', '#2196F3'),
               ('avg_chrf', 'ChrF (higher=better)', '#4CAF50'),
               ('avg_rtf',  'RTF (lower=faster)', '#FF9800'),
               ('params_M', 'Parameters (M)', '#9C27B0')]
    for ax, (key, title, color) in zip(axes.flat, metrics):
        vals = [s.get(key, 0) for s in data]
        bars = ax.bar(range(len(labels)), vals, color=color, alpha=0.85, edgecolor='white')
        ax.set_title(title, fontweight='bold')
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), f'{v:.1f}',
                    ha='center', va='bottom', fontsize=8)
    plt.tight_layout(); plt.savefig(f'{FIG_DIR}/{save_name}'); plt.show()

def plot_size_vs_quality(summaries=None, save_name='size_vs_quality.png'):
    data = summaries or ALL_SUMMARIES
    if not data: return
    fig, ax = plt.subplots(figsize=(10, 7))
    params = [s['params_M'] for s in data]
    bleu = [s['avg_bleu'] for s in data]
    chrf = [s['avg_chrf'] for s in data]
    ax.scatter(params, bleu, s=120, c='#2196F3', zorder=5, label='BLEU')
    ax.scatter(params, chrf, s=120, c='#4CAF50', marker='s', zorder=5, label='ChrF')
    for i, lbl in enumerate([s['label'] for s in data]):
        ax.annotate(lbl, (params[i], bleu[i]), fontsize=7, xytext=(5,5), textcoords='offset points')
    ax.set_xlabel('Parameters (M)'); ax.set_ylabel('Score')
    ax.set_title('Model Size vs Translation Quality', fontweight='bold')
    ax.legend(); plt.tight_layout(); plt.savefig(f'{FIG_DIR}/{save_name}'); plt.show()

def plot_layer_scores(scores_dict, title='Layer Importance', save_name=None):
    indices = sorted(scores_dict.keys())
    vals = [scores_dict[i] for i in indices]
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['#d32f2f' if v > np.percentile(vals, 75) else '#ff9800' if v > np.percentile(vals, 50) else '#4caf50' if v > np.percentile(vals, 25) else '#90caf9' for v in vals]
    ax.bar(indices, vals, color=colors, edgecolor='white')
    ax.set_xlabel('Layer Index'); ax.set_ylabel('Importance')
    ax.set_title(title, fontweight='bold'); ax.set_xticks(indices)
    plt.tight_layout()
    if save_name: plt.savefig(f'{FIG_DIR}/{save_name}')
    plt.show()

print('Plotting helpers ready.')

---
# Phase 0: Baseline Benchmark
Load the full teacher model, measure size and translation quality.

In [ ]:
model, processor = load_base_model()
baseline_breakdown = print_model_breakdown(model, 'Baseline Model')

In [ ]:
from datasets import load_dataset

TARGET_LANG = 'ben'
FLEURS_SRC, FLEURS_TGT = 'en_us', 'bn_in'
N_EVAL = 20

print(f'Loading FLEURS: {FLEURS_SRC} to {FLEURS_TGT}, {N_EVAL} samples...')
ds_src = load_dataset('google/fleurs', FLEURS_SRC, split='test', trust_remote_code=True)
ds_tgt = load_dataset('google/fleurs', FLEURS_TGT, split='test', trust_remote_code=True)
src_by_id = {ex['id']: ex for ex in ds_src}
tgt_by_id = {ex['id']: ex for ex in ds_tgt}
common_ids = sorted(set(src_by_id) & set(tgt_by_id))[:N_EVAL]

eval_samples = []
for sid in common_ids:
    wav = np.array(src_by_id[sid]['audio']['array'], dtype=np.float32)
    sr = src_by_id[sid]['audio']['sampling_rate']
    if sr != 16000:
        wav = torchaudio.functional.resample(torch.tensor(wav), sr, 16000).numpy()
    eval_samples.append(dict(id=sid, wav=wav, ref=tgt_by_id[sid]['transcription'],
                             en_text=src_by_id[sid]['transcription']))
print(f'Loaded {len(eval_samples)} eval samples.')

In [ ]:
baseline_ckpt = load_latest_checkpoint('phase0_baseline')
if baseline_ckpt:
    baseline_results = baseline_ckpt['results']
    baseline_summary = baseline_ckpt['summary']
    print(f'Loaded baseline: BLEU={baseline_summary["avg_bleu"]:.2f}')
else:
    baseline_results, baseline_summary = run_benchmark(
        model, eval_samples, label='P0_Baseline', save_n=3)
    save_checkpoint(dict(results=baseline_results, summary=baseline_summary,
                         breakdown=baseline_breakdown), name='phase0_baseline', step=0)

store_summary(baseline_summary)
plot_phase_comparison()

---
# Phase 1: Vocabulary / Embedding Pruning
**Paper:** Asahi et al. (EMNLP 2023)

NLLB vocabulary has **256,102 tokens** for ~100 languages.
We keep only ~5-7 languages. Trimming saves ~200M params with near-zero quality impact.

In [ ]:
def identify_used_tokens(proc, target_lang_codes, n_corpus=2000):
    from datasets import load_dataset
    fleurs_codes = dict(eng='en_us', ben='bn_in', cmn='cmn_hans_cn',
                        fra='fr_fr', deu='de_de', hin='hi_in', urd='ur_pk')
    used = set()
    tok = proc.tokenizer
    if hasattr(tok, 'all_special_ids'): used.update(tok.all_special_ids)
    for tid in range(len(tok)):
        t = tok.convert_ids_to_tokens(tid)
        if t and t.startswith('__') and t.endswith('__'): used.add(tid)
    for lang, fc in fleurs_codes.items():
        if lang not in target_lang_codes: continue
        print(f'  Scanning {lang} ({fc})...')
        try:
            ds = load_dataset('google/fleurs', fc, split='train', trust_remote_code=True)
            for i, ex in enumerate(ds):
                if i >= n_corpus: break
                text = ex.get('transcription', '')
                if text: used.update(tok.encode(text, add_special_tokens=False))
        except Exception as e:
            print(f'    Warning: {lang}: {e}')
    print(f'  Unique tokens: {len(used)} / {len(tok)}')
    return sorted(used)

def trim_vocabulary(mdl, proc, keep_ids):
    keep_t = torch.tensor(keep_ids, dtype=torch.long)
    old_v = len(proc.tokenizer)
    new_v = len(keep_ids)
    print(f'  Vocabulary: {old_v} to {new_v} ({new_v/old_v*100:.1f}%)')
    trimmed = 0
    for name, module in mdl.named_modules():
        if isinstance(module, nn.Embedding) and module.num_embeddings >= old_v * 0.9:
            new_w = module.weight.data[keep_t].clone()
            new_emb = nn.Embedding(new_v, module.embedding_dim, padding_idx=module.padding_idx)
            new_emb.weight.data = new_w
            parts = name.split('.')
            parent = mdl
            for p in parts[:-1]: parent = getattr(parent, p)
            setattr(parent, parts[-1], new_emb)
            trimmed += 1
            print(f'  Trimmed embedding: {name}')
        elif isinstance(module, nn.Linear) and module.out_features >= old_v * 0.9:
            new_w = module.weight.data[keep_t].clone()
            new_lin = nn.Linear(module.in_features, new_v, bias=module.bias is not None)
            new_lin.weight.data = new_w
            if module.bias is not None: new_lin.bias.data = module.bias.data[keep_t].clone()
            parts = name.split('.')
            parent = mdl
            for p in parts[:-1]: parent = getattr(parent, p)
            setattr(parent, parts[-1], new_lin)
            trimmed += 1
            print(f'  Trimmed projection: {name}')
    if trimmed == 0: print('  WARNING: No layers trimmed.')
    return mdl

print('Vocab trimming functions ready.')

In [ ]:
try:
    model_p1, processor = load_model_from_drive('phase1_vocab_trimmed')
    print('Loaded Phase 1 model from Drive.')
except:
    print('Running vocab trimming...')
    TARGET_LANGS = ['eng', 'ben', 'cmn', 'fra', 'deu', 'hin', 'urd']
    keep_ids = identify_used_tokens(processor, TARGET_LANGS)
    pre = count_params(model)
    model = trim_vocabulary(model, processor, keep_ids)
    post = count_params(model)
    print(f'  Params: {pre:.1f}M to {post:.1f}M (saved {pre-post:.1f}M)')
    save_checkpoint(dict(keep_ids=keep_ids, pre=pre, post=post), name='phase1_vocab', step=0)
    save_model_to_drive(model, processor, 'phase1_vocab_trimmed')
    model_p1 = model

print_model_breakdown(model_p1, 'After Phase 1: Vocab Trimmed')

In [ ]:
p1_ckpt = load_latest_checkpoint('phase1_benchmark')
if p1_ckpt:
    p1_results, p1_summary = p1_ckpt['results'], p1_ckpt['summary']
else:
    p1_results, p1_summary = run_benchmark(model_p1, eval_samples, label='P1_VocabTrim', save_n=2)
    save_checkpoint(dict(results=p1_results, summary=p1_summary), name='phase1_benchmark', step=0)
store_summary(p1_summary)
plot_phase_comparison()

---
# Phase 2: Text Encoder Removal
For S2S, the pipeline is: Speech Encoder -> Text Decoder -> T2U -> Vocoder.
The text_encoder is only used for T2T/T2S tasks. We verify and remove it.

In [ ]:
def check_text_encoder_used(mdl):
    if not hasattr(mdl, 'text_encoder') or mdl.text_encoder is None:
        print('  text_encoder not present.'); return False
    called = [False]
    def hook(mod, inp, out): called[0] = True
    h = mdl.text_encoder.register_forward_hook(hook)
    try: _ = run_s2st(mdl, np.random.randn(48000).astype(np.float32), 'ben')
    except: pass
    h.remove()
    return called[0]

def remove_text_encoder(mdl):
    if hasattr(mdl, 'text_encoder') and mdl.text_encoder is not None:
        enc_p = count_params(mdl.text_encoder)
        del mdl.text_encoder; mdl.text_encoder = None
        gc.collect(); torch.cuda.empty_cache()
        print(f'  Removed text_encoder ({enc_p:.1f}M params)')
    return mdl

print('Text encoder analysis ready.')

In [ ]:
try:
    model_p2, processor = load_model_from_drive('phase2_no_text_enc')
    print('Loaded Phase 2 from Drive.')
except:
    model_p2 = model_p1
    used = check_text_encoder_used(model_p2)
    print(f'  text_encoder used in S2S: {used}')
    if not used:
        print('  Safe to remove for S2S-only.')
        model_p2 = remove_text_encoder(model_p2)
    else:
        print('  text_encoder IS used. Keeping 8 of 24 layers.')
        layers = model_p2.text_encoder.layers
        keep = list(range(4)) + list(range(len(layers)-4, len(layers)))
        model_p2.text_encoder.layers = nn.ModuleList([layers[i] for i in keep])
    save_model_to_drive(model_p2, processor, 'phase2_no_text_enc')

print_model_breakdown(model_p2, 'After Phase 2: Text Encoder Removed')

In [ ]:
p2_ckpt = load_latest_checkpoint('phase2_benchmark')
if p2_ckpt:
    p2_results, p2_summary = p2_ckpt['results'], p2_ckpt['summary']
else:
    p2_results, p2_summary = run_benchmark(model_p2, eval_samples, label='P2_NoTextEnc', save_n=2)
    save_checkpoint(dict(results=p2_results, summary=p2_summary), name='phase2_benchmark', step=0)
store_summary(p2_summary)
plot_phase_comparison()

---
# Phase 3: Text Decoder Iterative Layer Pruning
**Paper:** Moslem (IWSLT 2025), CULL-MT (2024)

Iterative greedy pruning: remove one layer at a time, evaluate ChrF, repeat.
Target: remove 6-8 of 24 text decoder layers.

In [ ]:
def find_layers_attr(component):
    for attr in ['layers', 'layer', 'inner_layers', 'encoder_layers', 'decoder_layers']:
        if hasattr(component, attr): return attr
    return None

def iterative_layer_prune(mdl, component_name, samples, n_remove,
                          tgt_lang='ben', max_eval=10):
    """
    Iterative greedy layer pruning (Moslem, IWSLT 2025).
    Removes one layer per iteration, picking the one whose removal
    causes the least ChrF degradation.
    """
    parent = getattr(mdl, component_name)
    layers_attr = find_layers_attr(parent)
    if layers_attr is None:
        print(f'  No layers found on {component_name}'); return [], []
    current = list(getattr(parent, layers_attr))
    orig_indices = list(range(len(current)))
    removed, log = [], []

    for it in range(n_remove):
        print(f'\n  Iter {it+1}/{n_remove} ({len(current)} layers remain)')
        scores = {}
        for idx in range(len(current)):
            temp = current[:idx] + current[idx+1:]
            setattr(parent, layers_attr, nn.ModuleList(temp))
            sc = quick_eval_chrf(mdl, samples, tgt_lang, max_eval)
            scores[idx] = (orig_indices[idx], sc)
            print(f'    Remove L{orig_indices[idx]:>2} -> ChrF={sc:.2f}')
        setattr(parent, layers_attr, nn.ModuleList(current))

        best_idx = max(scores, key=lambda k: scores[k][1])
        best_orig, best_sc = scores[best_idx]
        current.pop(best_idx)
        orig_indices.pop(best_idx)
        setattr(parent, layers_attr, nn.ModuleList(current))
        removed.append(best_orig)
        log.append(dict(iter=it+1, removed=best_orig, chrf=best_sc,
                        remaining=len(current)))
        print(f'  -> Removed layer {best_orig} (ChrF={best_sc:.2f})')

    return removed, log

print('iterative_layer_prune() ready.')

In [ ]:
N_DEC_REMOVE = 6

p3_ckpt = load_latest_checkpoint('phase3_dec_pruning')
if p3_ckpt:
    removed_dec = p3_ckpt['removed']; p3_log = p3_ckpt['log']
    print(f'Loaded Phase 3: removed {removed_dec}')
    try: model_p3, processor = load_model_from_drive('phase3_dec_pruned')
    except:
        model_p3 = model_p2
        parent = model_p3.text_decoder
        la = find_layers_attr(parent)
        cur = list(getattr(parent, la))
        keep = [i for i in range(len(cur)) if i not in removed_dec]
        setattr(parent, la, nn.ModuleList([cur[i] for i in keep]))
else:
    print(f'Running Phase 3: decoder pruning ({N_DEC_REMOVE} layers)...')
    model_p3 = model_p2
    removed_dec, p3_log = iterative_layer_prune(
        model_p3, 'text_decoder', eval_samples, N_DEC_REMOVE, TARGET_LANG)
    save_checkpoint(dict(removed=removed_dec, log=p3_log), name='phase3_dec_pruning', step=0)
    save_model_to_drive(model_p3, processor, 'phase3_dec_pruned')

print(f'Decoder layers removed: {removed_dec}')
print_model_breakdown(model_p3, 'After Phase 3: Decoder Pruned')

In [ ]:
if p3_log:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    iters = [e['iter'] for e in p3_log]
    chrfs = [e['chrf'] for e in p3_log]
    ax1.plot(iters, chrfs, 'o-', color='#4CAF50', lw=2, ms=8)
    for e in p3_log: ax1.annotate(f'L{e["removed"]}', (e['iter'], e['chrf']), fontsize=8, ha='center', va='bottom')
    ax1.set_xlabel('Iteration'); ax1.set_ylabel('ChrF'); ax1.set_title('Decoder: ChrF After Each Removal', fontweight='bold')
    ax2.bar(iters, [e['remaining'] for e in p3_log], color='#9C27B0', alpha=0.8)
    ax2.set_xlabel('Iteration'); ax2.set_ylabel('Layers'); ax2.set_title('Decoder Layers Remaining', fontweight='bold')
    plt.tight_layout(); plt.savefig(f'{FIG_DIR}/phase3_dec.png'); plt.show()

In [ ]:
p3b = load_latest_checkpoint('phase3_benchmark')
if p3b: p3_results, p3_summary = p3b['results'], p3b['summary']
else:
    p3_results, p3_summary = run_benchmark(model_p3, eval_samples, label='P3_DecPrune', save_n=2)
    save_checkpoint(dict(results=p3_results, summary=p3_summary), name='phase3_benchmark', step=0)
store_summary(p3_summary); plot_phase_comparison()

---
# Phase 4: Speech Encoder Iterative Layer Pruning
**Paper:** ShortGPT (ACL 2025) for Block Influence; Moslem (IWSLT 2025) for iterative greedy

Target: remove 6-8 of 24 speech encoder layers.

In [ ]:
def compute_block_influence(mdl, samples, component_name='speech_encoder', max_n=50):
    parent = getattr(mdl, component_name)
    la = find_layers_attr(parent)
    if la is None: return {}
    layers = getattr(parent, la)
    bi = {i: [] for i in range(len(layers))}
    hooks = []
    for i, layer in enumerate(layers):
        def make_hook(idx):
            def hook(mod, inp, out):
                x = inp[0].detach().float()
                y = (out[0] if isinstance(out, tuple) else out).detach().float()
                xf = x.reshape(x.shape[0], -1)
                yf = y.reshape(y.shape[0], -1)
                cos = F.cosine_similarity(xf, yf, dim=-1)
                bi[idx].append((1 - cos).mean().item())
            return hook
        hooks.append(layer.register_forward_hook(make_hook(i)))
    for idx, s in enumerate(samples[:max_n]):
        if idx % 10 == 0: print(f'  Calibrating {idx}/{min(max_n, len(samples))}...')
        try:
            inputs = processor(audios=s['wav'], sampling_rate=16000, return_tensors='pt')
            inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
            with torch.no_grad(): mdl.generate(**inputs, tgt_lang=TARGET_LANG, generate_speech=False)
        except: pass
    for h in hooks: h.remove()
    return {i: float(np.mean(v)) if v else 0 for i, v in bi.items()}

print('compute_block_influence() ready.')

In [ ]:
N_ENC_REMOVE = 6

p4_ckpt = load_latest_checkpoint('phase4_enc_pruning')
if p4_ckpt:
    removed_enc = p4_ckpt['removed']
    bi_scores = p4_ckpt.get('bi_scores', {})
    p4_log = p4_ckpt['log']
    print(f'Loaded Phase 4: removed {removed_enc}')
    try: model_p4, processor = load_model_from_drive('phase4_enc_pruned')
    except:
        model_p4 = model_p3
        parent = model_p4.speech_encoder
        la = find_layers_attr(parent)
        cur = list(getattr(parent, la))
        keep = [i for i in range(len(cur)) if i not in removed_enc]
        setattr(parent, la, nn.ModuleList([cur[i] for i in keep]))
else:
    model_p4 = model_p3
    print('Step 1: Block Influence scores...')
    bi_scores = compute_block_influence(model_p4, eval_samples)
    plot_layer_scores(bi_scores, 'Speech Encoder Block Influence', 'phase4_bi.png')
    print(f'\nStep 2: Iterative pruning ({N_ENC_REMOVE} layers)...')
    removed_enc, p4_log = iterative_layer_prune(
        model_p4, 'speech_encoder', eval_samples, N_ENC_REMOVE, TARGET_LANG)
    save_checkpoint(dict(removed=removed_enc, log=p4_log, bi_scores=bi_scores),
                    name='phase4_enc_pruning', step=0)
    save_model_to_drive(model_p4, processor, 'phase4_enc_pruned')

print(f'Encoder layers removed: {removed_enc}')
print_model_breakdown(model_p4, 'After Phase 4: Encoder Pruned')

In [ ]:
if bi_scores and p4_log:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    indices = sorted(bi_scores.keys())
    vals = [bi_scores[i] for i in indices]
    colors = ['#d32f2f' if i in removed_enc else '#4caf50' for i in indices]
    ax1.bar(indices, vals, color=colors, edgecolor='white')
    ax1.set_title('Block Influence (red=removed)', fontweight='bold')
    ax1.set_xlabel('Layer'); ax1.set_xticks(indices)
    iters = [e['iter'] for e in p4_log]
    ax2.plot(iters, [e['chrf'] for e in p4_log], 'o-', color='#2196F3', lw=2)
    for e in p4_log: ax2.annotate(f'L{e["removed"]}', (e['iter'], e['chrf']), fontsize=8)
    ax2.set_title('ChrF After Each Removal', fontweight='bold'); ax2.set_xlabel('Iteration')
    plt.tight_layout(); plt.savefig(f'{FIG_DIR}/phase4_enc.png'); plt.show()

In [ ]:
p4b = load_latest_checkpoint('phase4_benchmark')
if p4b: p4_results, p4_summary = p4b['results'], p4b['summary']
else:
    p4_results, p4_summary = run_benchmark(model_p4, eval_samples, label='P4_EncPrune', save_n=2)
    save_checkpoint(dict(results=p4_results, summary=p4_summary), name='phase4_benchmark', step=0)
store_summary(p4_summary); plot_phase_comparison()

---
# Phase 5: Width Pruning (FLAP)
**Paper:** FLAP (AAAI 2024)

Structurally remove FFN neurons (shrink weight matrices). Creates smaller dense
matrices for real GPU speedup, unlike zeroing which keeps full-size matrices.

In [ ]:
def find_ffn_pair(layer):
    pairs = [('fc1','fc2'), ('intermediate_dense','output_dense'), ('linear1','linear2')]
    for n1, n2 in pairs:
        if hasattr(layer, n1) and hasattr(layer, n2): return n1, n2
    for cname, child in layer.named_children():
        for n1, n2 in pairs:
            if hasattr(child, n1) and hasattr(child, n2): return f'{cname}.{n1}', f'{cname}.{n2}'
    return None, None

def get_nested(obj, path):
    for p in path.split('.'): obj = getattr(obj, p)
    return obj

def set_nested(obj, path, val):
    parts = path.split('.')
    for p in parts[:-1]: obj = getattr(obj, p)
    setattr(obj, parts[-1], val)

def structural_prune_ffn(layer, fc1_name, fc2_name, ratio=0.25):
    fc1 = get_nested(layer, fc1_name)
    fc2 = get_nested(layer, fc2_name)
    w = fc1.weight.data.float()
    scores = w.abs().mean(dim=1)
    if fc1.bias is not None: scores += fc1.bias.data.float().abs()
    n_total = len(scores)
    n_keep = max(int(n_total * (1 - ratio)), 64)
    _, keep = torch.topk(scores, n_keep)
    keep = keep.sort().values
    new1 = nn.Linear(fc1.in_features, n_keep, bias=fc1.bias is not None)
    new1.weight.data = fc1.weight.data[keep].clone()
    if fc1.bias is not None: new1.bias.data = fc1.bias.data[keep].clone()
    new2 = nn.Linear(n_keep, fc2.out_features, bias=fc2.bias is not None)
    new2.weight.data = fc2.weight.data[:, keep].clone()
    if fc2.bias is not None: new2.bias.data = fc2.bias.data.clone()
    new1 = new1.to(fc1.weight.device, fc1.weight.dtype)
    new2 = new2.to(fc2.weight.device, fc2.weight.dtype)
    set_nested(layer, fc1_name, new1)
    set_nested(layer, fc2_name, new2)
    return n_total, n_keep

def apply_width_pruning(mdl, comp_name, ratio=0.25):
    parent = getattr(mdl, comp_name)
    la = find_layers_attr(parent)
    if la is None: print(f'  Skip {comp_name}'); return []
    layers = getattr(parent, la)
    stats = []
    for i, layer in enumerate(layers):
        n1, n2 = find_ffn_pair(layer)
        if n1 is None: continue
        nt, nk = structural_prune_ffn(layer, n1, n2, ratio)
        stats.append(dict(layer=i, total=nt, kept=nk))
        if i % 4 == 0: print(f'    L{i:>2}: {nt} -> {nk} ({nk/nt*100:.0f}%)')
    return stats

print('Width pruning functions ready.')

In [ ]:
FFN_PRUNE_RATIO = 0.25

try:
    model_p5, processor = load_model_from_drive('phase5_width_pruned')
    print('Loaded Phase 5 from Drive.')
except:
    print(f'Running Phase 5: width pruning at {FFN_PRUNE_RATIO*100:.0f}%...')
    model_p5 = model_p4
    pre = count_params(model_p5)
    stats = {}
    for comp in ['speech_encoder', 'text_decoder']:
        print(f'\n  Pruning {comp}...')
        stats[comp] = apply_width_pruning(model_p5, comp, FFN_PRUNE_RATIO)
    if hasattr(model_p5, 't2u_model') and model_p5.t2u_model is not None:
        print(f'\n  Pruning t2u_model...')
        stats['t2u_model'] = apply_width_pruning(model_p5, 't2u_model', FFN_PRUNE_RATIO)
    post = count_params(model_p5)
    print(f'\n  Width pruning: {pre:.1f}M to {post:.1f}M (saved {pre-post:.1f}M)')
    save_checkpoint(dict(stats=stats, ratio=FFN_PRUNE_RATIO), name='phase5_width', step=0)
    save_model_to_drive(model_p5, processor, 'phase5_width_pruned')

print_model_breakdown(model_p5, 'After Phase 5: Width Pruned')

In [ ]:
p5b = load_latest_checkpoint('phase5_benchmark')
if p5b: p5_results, p5_summary = p5b['results'], p5b['summary']
else:
    p5_results, p5_summary = run_benchmark(model_p5, eval_samples, label='P5_WidthPrune', save_n=2)
    save_checkpoint(dict(results=p5_results, summary=p5_summary), name='phase5_benchmark', step=0)
store_summary(p5_summary); plot_phase_comparison()

---
# Phase 6: T2U Model Pruning
Apply middle-layer removal to T2U transformer stacks (2-3 layers per stack).

In [ ]:
try:
    model_p6, processor = load_model_from_drive('phase6_t2u_pruned')
    print('Loaded Phase 6 from Drive.')
except:
    model_p6 = model_p5
    if hasattr(model_p6, 't2u_model') and model_p6.t2u_model is not None:
        t2u = model_p6.t2u_model
        pre = count_params(t2u)
        for attr_name in dir(t2u):
            attr = getattr(t2u, attr_name, None)
            if isinstance(attr, nn.ModuleList) and len(attr) > 3:
                n_rm = min(2, len(attr) - 2)
                mid = len(attr) // 2
                rm_idx = list(range(mid - n_rm//2, mid + (n_rm+1)//2))
                keep = [i for i in range(len(attr)) if i not in rm_idx]
                setattr(t2u, attr_name, nn.ModuleList([attr[i] for i in keep]))
                print(f'  {attr_name}: removed {rm_idx}, kept {len(keep)}')
        post = count_params(t2u)
        print(f'  T2U: {pre:.1f}M to {post:.1f}M')
    else: print('  No T2U model.')
    save_model_to_drive(model_p6, processor, 'phase6_t2u_pruned')

print_model_breakdown(model_p6, 'After Phase 6: T2U Pruned')

In [ ]:
p6b = load_latest_checkpoint('phase6_benchmark')
if p6b: p6_results, p6_summary = p6b['results'], p6b['summary']
else:
    p6_results, p6_summary = run_benchmark(model_p6, eval_samples, label='P6_T2UPrune', save_n=2)
    save_checkpoint(dict(results=p6_results, summary=p6_summary), name='phase6_benchmark', step=0)
store_summary(p6_summary); plot_phase_comparison(); plot_size_vs_quality()

---
# Phase 7: Recovery Fine-tuning with LoRA
**Paper:** Moslem (IWSLT 2025)

Fine-tune using S2TT cross-entropy loss + LoRA for memory efficiency.

In [ ]:
from datasets import load_dataset as ld
print('Loading FLEURS training data...')
ft_src = ld('google/fleurs', FLEURS_SRC, split='train', trust_remote_code=True)
ft_tgt = ld('google/fleurs', FLEURS_TGT, split='train', trust_remote_code=True)
fs = {ex['id']: ex for ex in ft_src}
ft = {ex['id']: ex for ex in ft_tgt}
ft_samples = []
for sid in sorted(set(fs) & set(ft)):
    wav = np.array(fs[sid]['audio']['array'], dtype=np.float32)
    sr = fs[sid]['audio']['sampling_rate']
    if sr != 16000: wav = torchaudio.functional.resample(torch.tensor(wav), sr, 16000).numpy()
    ft_samples.append(dict(wav=wav[:16000*15], ref=ft[sid]['transcription']))
print(f'Loaded {len(ft_samples)} training pairs.')

In [ ]:
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW
import random

MAX_STEPS, LOG_EVERY, SAVE_EVERY, LR = 2000, 50, 500, 1e-5

ft_ckpt = load_latest_checkpoint('phase7_ft')
start_step = ft_ckpt['step'] if ft_ckpt else 0

if start_step >= MAX_STEPS:
    print(f'Fine-tuning complete at step {start_step}.')
    try: model_p7, processor = load_model_from_drive('phase7_finetuned')
    except: model_p7 = model_p6
else:
    model_p7 = model_p6
    targets = set()
    for name, mod in model_p7.named_modules():
        if isinstance(mod, nn.Linear):
            sn = name.split('.')[-1]
            if sn in ['q_proj','k_proj','v_proj','out_proj','fc1','fc2']:
                targets.add(sn)
    targets = list(targets) or ['q_proj', 'v_proj']
    print(f'  LoRA targets: {targets}')

    try:
        model_p7 = get_peft_model(model_p7, LoraConfig(
            r=16, lora_alpha=32, target_modules=targets, lora_dropout=0.05, bias='none'))
        tr = sum(p.numel() for p in model_p7.parameters() if p.requires_grad)/1e6
        print(f'  LoRA trainable: {tr:.1f}M')
    except Exception as e:
        print(f'  LoRA failed ({e}), unfreezing speech_encoder only')
        for p in model_p7.parameters(): p.requires_grad = False
        for p in model_p7.speech_encoder.parameters(): p.requires_grad = True

    opt = AdamW([p for p in model_p7.parameters() if p.requires_grad], lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, MAX_STEPS, eta_min=1e-7)
    if ft_ckpt and 'opt' in ft_ckpt:
        try: opt.load_state_dict(ft_ckpt['opt'])
        except: pass

    model_p7.train()
    loss_log = ft_ckpt.get('loss_log', []) if ft_ckpt else []
    step = start_step
    random.shuffle(ft_samples)
    print(f'\n  Starting from step {step}...')

    while step < MAX_STEPS:
        s = ft_samples[step % len(ft_samples)]
        try:
            inputs = processor(audios=s['wav'], sampling_rate=16000, return_tensors='pt')
            inputs = {k: v.to(model_p7.device) for k, v in inputs.items()}
            labels = processor.tokenizer(text=s['ref'], return_tensors='pt',
                                        padding=True).input_ids.to(model_p7.device)
            opt.zero_grad()
            with torch.amp.autocast('cuda', dtype=torch.float16):
                loss = model_p7(**inputs, labels=labels).loss
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_p7.parameters(), 1.0)
            opt.step(); sched.step()
            loss_log.append(loss.item()); step += 1
            if step % LOG_EVERY == 0:
                print(f'  Step {step:>5}/{MAX_STEPS}  loss={np.mean(loss_log[-LOG_EVERY:]):.4f}  lr={sched.get_last_lr()[0]:.2e}')
            if step % SAVE_EVERY == 0:
                save_checkpoint(dict(step=step, loss_log=loss_log, opt=opt.state_dict()),
                                name='phase7_ft', step=step)
        except RuntimeError as e:
            if 'out of memory' in str(e): torch.cuda.empty_cache(); continue
            raise

    model_p7.eval()
    if hasattr(model_p7, 'merge_and_unload'):
        print('  Merging LoRA weights...')
        model_p7 = model_p7.merge_and_unload()
    print(f'  Done. Final loss: {np.mean(loss_log[-50:]):.4f}')
    save_model_to_drive(model_p7, processor, 'phase7_finetuned')

In [ ]:
ft_ckpt = load_latest_checkpoint('phase7_ft')
if ft_ckpt and 'loss_log' in ft_ckpt:
    losses = ft_ckpt['loss_log']
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(losses, alpha=0.3, color='steelblue', lw=0.5, label='Raw')
    ema, val = [], losses[0]
    for l in losses: val = 0.02*l + 0.98*val; ema.append(val)
    ax.plot(ema, color='firebrick', lw=2, label='EMA')
    ax.set_xlabel('Step'); ax.set_ylabel('Loss')
    ax.set_title('Phase 7: Fine-tuning Loss', fontweight='bold')
    ax.legend(); plt.tight_layout(); plt.savefig(f'{FIG_DIR}/phase7_loss.png'); plt.show()

In [ ]:
p7b = load_latest_checkpoint('phase7_benchmark')
if p7b: p7_results, p7_summary = p7b['results'], p7b['summary']
else:
    p7_results, p7_summary = run_benchmark(model_p7, eval_samples, label='P7_FineTuned', save_n=3)
    save_checkpoint(dict(results=p7_results, summary=p7_summary), name='phase7_benchmark', step=0)
store_summary(p7_summary); plot_phase_comparison()

---
# Phase 8: Final Results + Paper Table

In [ ]:
sc = load_latest_checkpoint('all_summaries')
if sc and 'summaries' in sc: ALL_SUMMARIES = sc['summaries']

print('\n' + '='*80)
print('  FINAL: SeamlessM4T v2 Large  Structured Compression')
print('  Task: English to Bengali Speech Translation (FLEURS test)')
print('='*80)
hdr = f'{"Phase":<25} {"Params(M)":>10} {"Delta":>8} {"BLEU":>7} {"ChrF":>7} {"RTF":>7}'
print(hdr); print('-'*len(hdr))
bp = ALL_SUMMARIES[0]['params_M'] if ALL_SUMMARIES else 2300
for s in ALL_SUMMARIES:
    d = (1 - s['params_M']/bp)*100 if bp else 0
    ds = f'-{d:.1f}%' if d > 0 else 'base'
    print(f'  {s["label"]:<23} {s["params_M"]:>8.1f}  {ds:>7}  {s["avg_bleu"]:>6.2f}  {s["avg_chrf"]:>6.2f}  {s["avg_rtf"]:>6.4f}')
print('='*80)
if len(ALL_SUMMARIES) >= 2:
    f, b = ALL_SUMMARIES[-1], ALL_SUMMARIES[0]
    print(f'  Param reduction: {(1-f["params_M"]/b["params_M"])*100:.1f}%')
    print(f'  Speed (RTF): {b["avg_rtf"]/f["avg_rtf"]:.2f}x faster' if f['avg_rtf']>0 else '')

In [ ]:
if len(ALL_SUMMARIES) >= 2:
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle('SeamlessM4T Compression Pipeline Results', fontsize=16, fontweight='bold')
    labels = [s['label'] for s in ALL_SUMMARIES]
    x = range(len(labels))

    ax1 = fig.add_subplot(2, 3, 1)
    ps = [s['params_M'] for s in ALL_SUMMARIES]
    ax1.bar(x, ps, color='#9C27B0', alpha=0.85)
    ax1.set_ylabel('Params (M)'); ax1.set_title('Model Size', fontweight='bold')
    ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=40, ha='right', fontsize=7)

    ax2 = fig.add_subplot(2, 3, 2)
    ax2.plot(x, [s['avg_bleu'] for s in ALL_SUMMARIES], 'o-', color='#2196F3', lw=2)
    ax2.set_ylabel('BLEU'); ax2.set_title('BLEU (higher=better)', fontweight='bold')
    ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=40, ha='right', fontsize=7)

    ax3 = fig.add_subplot(2, 3, 3)
    ax3.plot(x, [s['avg_chrf'] for s in ALL_SUMMARIES], 's-', color='#4CAF50', lw=2)
    ax3.set_ylabel('ChrF'); ax3.set_title('ChrF (higher=better)', fontweight='bold')
    ax3.set_xticks(x); ax3.set_xticklabels(labels, rotation=40, ha='right', fontsize=7)

    ax4 = fig.add_subplot(2, 3, 4)
    ax4.bar(x, [s['avg_rtf'] for s in ALL_SUMMARIES], color='#FF9800', alpha=0.85)
    ax4.set_ylabel('RTF'); ax4.set_title('RTF (lower=faster)', fontweight='bold')
    ax4.set_xticks(x); ax4.set_xticklabels(labels, rotation=40, ha='right', fontsize=7)

    ax5 = fig.add_subplot(2, 3, 5)
    ax5.scatter(ps, [s['avg_bleu'] for s in ALL_SUMMARIES], s=100, c='#2196F3', label='BLEU')
    ax5.scatter(ps, [s['avg_chrf'] for s in ALL_SUMMARIES], s=100, c='#4CAF50', marker='s', label='ChrF')
    ax5.set_xlabel('Params (M)'); ax5.set_ylabel('Score')
    ax5.set_title('Size vs Quality', fontweight='bold'); ax5.legend(fontsize=8)

    ax6 = fig.add_subplot(2, 3, 6)
    bp = ALL_SUMMARIES[0]['params_M'] or 1
    bb = ALL_SUMMARIES[0]['avg_bleu'] or 1
    bc = ALL_SUMMARIES[0]['avg_chrf'] or 1
    comp = [(1-s['params_M']/bp)*100 for s in ALL_SUMMARIES]
    ax6.plot(comp, [s['avg_bleu']/bb*100 for s in ALL_SUMMARIES], 'o-', color='#2196F3', label='BLEU %')
    ax6.plot(comp, [s['avg_chrf']/bc*100 for s in ALL_SUMMARIES], 's-', color='#4CAF50', label='ChrF %')
    ax6.axhline(y=90, color='gray', ls='--', alpha=0.5)
    ax6.set_xlabel('Compression %'); ax6.set_ylabel('Quality Retention %')
    ax6.set_title('Compression vs Quality', fontweight='bold'); ax6.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/final_comprehensive.png', dpi=150); plt.show()
    subprocess.run(f'rclone sync {FIG_DIR} {GDRIVE_ROOT}/figures/', shell=True)
    print('All figures synced to Drive.')

In [ ]:
print('\nDone. All results saved to Drive.')
session_status()